# Preprocessing Pipeline — Human Study Data (Session 2)

This notebook documents the full data pipeline from raw participant JSON files to the cleaned pair cache used in all analyses.

**Pipeline stages:**
1. Raw JSON files (`evaluation/humans/by_participant/*.json`)
2. `load_human_data()` → qualifying participants + common question IDs
3. `export_response_tables.py` → `exports/responses_human.csv`
4. `build_pair_cache.py` → `exports/pair_cache_raw.parquet`
5. `build_cleaned_pair_cache()` → `exports/pair_cache_cleaned.parquet`

**Key question:** Why does the pair cache appear to have N=39 participants, while `responses_human.csv` and the `subject_correlation_matrix` figure both show N=40?

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'analysis'))

EXPORTS = ROOT / 'analysis/session2/exports'
PART_DIR = ROOT / 'evaluation/humans/by_participant'

print(f'ROOT     : {ROOT}')
print(f'EXPORTS  : {EXPORTS}')
print(f'PART_DIR : {PART_DIR}')

ROOT     : /home/david/Desktop/yuna/HPA
EXPORTS  : /home/david/Desktop/yuna/HPA/analysis/session2/exports
PART_DIR : /home/david/Desktop/yuna/HPA/evaluation/humans/by_participant


## Stage 1 — Raw JSON Files

One JSON file per participant, stored in `evaluation/humans/by_participant/`. Each file contains the participant code, answers, and metadata.

In [2]:
files = sorted(PART_DIR.glob('*.json'))
print(f'Total JSON files: {len(files)}')
print()

all_raw = []
for f in files:
    p = json.load(open(f))
    answers = p.get('answers', [])
    qids = {a['question_id'] for a in answers}
    variants = {(a['question_id'], a.get('variant', 'C')) for a in answers}
    all_raw.append({
        'code': p['code'],
        'n_answers': len(answers),
        'n_qids': len(qids),
        'n_qv_pairs': len(variants),
    })

raw_df = pd.DataFrame(all_raw)
print('Answer counts by participant:')
print(raw_df['n_qids'].value_counts().sort_index().rename('participants').to_frame())
print()
print(raw_df.to_string(index=False))

Total JSON files: 40

Answer counts by participant:
        participants
n_qids              
116               11
124                5
134               24

    code  n_answers  n_qids  n_qv_pairs
HB857QDA        348     116         348
8HLCRUY1        348     116         348
DPCN9S37        348     116         348
FLWP3ZF7        348     116         348
J09HHJXK        348     116         348
K8VD3TYB        348     116         348
MPOMQQNU        348     116         348
NVF556XP        348     116         348
Q4PLE2BH        348     116         348
ZEF2XJBX        348     116         348
O23790AV        348     116         348
JOJCF28V        372     124         372
Y2IBXIJX        372     124         372
ON7P7X1E        372     124         372
TRFMV4H2        372     124         372
J0OU3FTE        372     124         372
BEK0Y4W3        402     134         402
DYAN3RF8        402     134         402
FXDCPFHM        402     134         402
HGIX7HE0        402     134         402
IV

In [3]:
# Compute question-ID sets per participant
qid_sets = {}
for f in files:
    p = json.load(open(f))
    qid_sets[p['code']] = {a['question_id'] for a in p.get('answers', [])}

# Group participants by how many questions they answered
groups = {}
for code, qids in qid_sets.items():
    n = len(qids)
    groups.setdefault(n, []).append(code)

print('Participant batches (by question count):')
for n in sorted(groups):
    codes = groups[n]
    print(f'  {n} questions: {len(codes)} participants — {codes}')

print()

# Compute pairwise intersections between groups
set_by_n = {n: set.intersection(*[qid_sets[c] for c in codes]) for n, codes in groups.items()}
ns = sorted(set_by_n.keys())
print('Question-ID set intersections between batches:')
for i, n1 in enumerate(ns):
    for n2 in ns[i:]:
        inter = len(set_by_n[n1] & set_by_n[n2])
        print(f'  {n1}-batch ∩ {n2}-batch = {inter} questions')

print()
# Common intersection across all 40 participants
common_qids = set.intersection(*qid_sets.values())
print(f'Common questions across ALL {len(qid_sets)} participants: {len(common_qids)}')

Participant batches (by question count):
  116 questions: 11 participants — ['HB857QDA', '8HLCRUY1', 'DPCN9S37', 'FLWP3ZF7', 'J09HHJXK', 'K8VD3TYB', 'MPOMQQNU', 'NVF556XP', 'Q4PLE2BH', 'ZEF2XJBX', 'O23790AV']
  124 questions: 5 participants — ['JOJCF28V', 'Y2IBXIJX', 'ON7P7X1E', 'TRFMV4H2', 'J0OU3FTE']
  134 questions: 24 participants — ['BEK0Y4W3', 'DYAN3RF8', 'FXDCPFHM', 'HGIX7HE0', 'IV0PL083', 'OBRNN1JM', '0X4B4L3T', '0S9EIHUY', 'BTTWUQG4', 'FYXQSR7I', '1UB7WNAM', '3BN3ZBUX', '85M7GIDP', 'EN84KWE9', 'IM2K4RKB', 'IR1A0UYG', 'P3TH5F6I', 'QGIQYWZ6', '9PLAS8J9', 'DOF4GFV3', 'FG80C8QR', 'KS8J7LVP', 'NFZJ7A7N', 'SEO7XXP3']

Question-ID set intersections between batches:
  116-batch ∩ 116-batch = 116 questions
  116-batch ∩ 124-batch = 113 questions
  116-batch ∩ 134-batch = 113 questions
  124-batch ∩ 124-batch = 124 questions
  124-batch ∩ 134-batch = 124 questions
  134-batch ∩ 134-batch = 134 questions

Common questions across ALL 40 participants: 113


## Stage 2 — `load_human_data()` (via `utils/load_session.py`)

Loads all qualifying participants (≥ `min_answers` answers), translates Korean answers, flattens to DataFrame, and intersects to common question IDs.

- **`min_answers` default = 348** → all 40 participants qualify
- **`common_qids`** = intersection of all participants' question sets = **113 questions**
- DataFrame is filtered to `common_qids` only → 40 × 113 × 3 variants = 13,560 rows

In [4]:
from utils.load_session import load_human_data
from config import MIN_ANSWERS_DEFAULT

print(f'MIN_ANSWERS_DEFAULT = {MIN_ANSWERS_DEFAULT}')
print()

# Count qualifying participants without full load (fast check)
qualifying = [(f.name, len(json.load(open(f)).get('answers', [])))
              for f in files]
n_qualify = sum(1 for _, n in qualifying if n >= MIN_ANSWERS_DEFAULT)
print(f'Participants with >= {MIN_ANSWERS_DEFAULT} answers: {n_qualify}/{len(qualifying)}')
print()
print('All answer counts:', sorted([n for _, n in qualifying]))

MIN_ANSWERS_DEFAULT = 348

Participants with >= 348 answers: 40/40

All answer counts: [348, 348, 348, 348, 348, 348, 348, 348, 348, 348, 348, 372, 372, 372, 372, 372, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402, 402]


## Stage 3 — `responses_human.csv`

Generated by `analysis/session2/export_response_tables.py`. Contains one row per (participant, question_id, variant). Restricted to the 113 common question IDs.

In [5]:
h_csv = pd.read_csv(EXPORTS / 'responses_human.csv')

print('=== responses_human.csv ===')
print(f'Total rows       : {len(h_csv):,}')
print(f'Unique participants : {h_csv["participant"].nunique()}')
print(f'Unique question_ids: {h_csv["question_id"].nunique()}')
print(f'Unique variants    : {sorted(h_csv["variant"].unique())}')
print(f'Shape should be    : 40 × 113 × 3 = {40*113*3:,}')
print()

# Per-participant question counts
per_p = h_csv.groupby('participant')['question_id'].nunique()
print(f'Questions per participant: min={per_p.min()}, max={per_p.max()}')
print(f'All have 113? {(per_p == 113).all()}')

=== responses_human.csv ===
Total rows       : 13,560
Unique participants : 40
Unique question_ids: 113
Unique variants    : ['A', 'B', 'C']
Shape should be    : 40 × 113 × 3 = 13,560

Questions per participant: min=113, max=113
All have 113? True


## Stage 4a — `check_completeness()` in `build_pair_cache.py`

Before building pairs, `build_pair_cache.py` calls `check_completeness()` on `responses_human.csv`. This function drops any participant with fewer questions than the maximum across participants.

Since all 40 participants have exactly 113 questions in `responses_human.csv`, none are dropped.

In [6]:
from utils.completeness import check_completeness

h_filtered, report = check_completeness(
    h_csv,
    group_col='participant',
    question_col='question_id',
    answer_col='response',
    label='Human responses',
    verbose=True,
)

print(f'After check_completeness: {h_filtered["participant"].nunique()} participants')
print(f'Flagged: {report["groups_flagged"]}')

[Human responses] Completeness check — participant, expected 113 questions
──────────────────────────────────────────────────────────────────────────
  ✓ All 40 participants complete.

After check_completeness: 40 participants
Flagged: []


## Stage 4b — Building `rater_answers` (free-text only)

`build_pair_cache.py` then filters to free-text question IDs only (`answer_type == 'text'`, excludes yes/no questions), and builds per-rater answer dictionaries. Participants with an **empty response** for a given (question, variant) are simply skipped for that pair — they are not dropped globally.

In [7]:
import json

q_meta = json.load(open(ROOT / 'experiment/s2_v4/s4_question.json'))
answer_type_map = {q['question_id']: q['answer_type'] for q in q_meta}

all_qids_in_csv = set(h_csv['question_id'].astype(int).unique())
text_qids = {qid for qid in all_qids_in_csv if answer_type_map.get(qid) == 'text'}
yesno_qids = all_qids_in_csv - text_qids

print(f'Total question IDs in CSV: {len(all_qids_in_csv)}')
print(f'Free-text questions: {len(text_qids)}')
print(f'Yes/no questions excluded: {len(yesno_qids)}')
print()

# Simulate rater_answers building for human participants
rater_answers = {}
for _, row in h_filtered[h_filtered['question_id'].isin(text_qids)].iterrows():
    pid = row['participant']
    qid = int(row['question_id'])
    var = row['variant']
    ans = str(row['response'] or '').strip()
    if not ans:
        continue
    if pid not in rater_answers:
        rater_answers[pid] = {}
    rater_answers[pid][(qid, var)] = ans

print(f'Participants with at least one answer in rater_answers: {len(rater_answers)}')
print()

# Check coverage per participant
expected = len(text_qids) * 3  # all q-v pairs
incomplete = {}
for pid, answers in rater_answers.items():
    if len(answers) < expected:
        incomplete[pid] = (len(answers), expected - len(answers))

if incomplete:
    print('Participants with < full coverage in rater_answers:')
    for pid, (n, miss) in sorted(incomplete.items(), key=lambda x: x[1][0]):
        print(f'  {pid}: {n}/{expected} q-v pairs (missing {miss})')
else:
    print('All participants have full q-v coverage.')

Total question IDs in CSV: 113
Free-text questions: 88
Yes/no questions excluded: 25

Participants with at least one answer in rater_answers: 40

All participants have full q-v coverage.


## Stage 4c — `pair_cache_raw.parquet`

Pairs are formed between all raters (human-human, human-model, model-model). The pair ordering is **lexicographic**: `subject_1 = min(r1, r2)`, `subject_2 = max(r1, r2)`.

**Important:** this means the lexicographically *first* participant always ends up in `subject_1`, and the lexicographically *last* participant always ends up in `subject_2`. As a result:
- `hh['subject_1'].nunique()` underestimates unique humans (misses the lex-last participant)
- `hh['subject_2'].nunique()` underestimates unique humans (misses the lex-first participant)
- The **union** of both columns gives the correct N=40

In [8]:
pair_raw = pd.read_parquet(EXPORTS / 'pair_cache_raw.parquet')
hh_raw = pair_raw[pair_raw['pair_type'] == 'HH']

print('=== pair_cache_raw (HH pairs only) ===')
print(f'Total HH pairs: {len(hh_raw):,}')
print()

s1_raw = set(hh_raw['subject_1'].unique())
s2_raw = set(hh_raw['subject_2'].unique())
union_raw = s1_raw | s2_raw

print(f'subject_1.nunique()         : {len(s1_raw)}  (naive count — misleading!)')
print(f'subject_2.nunique()         : {len(s2_raw)}  (naive count — misleading!)')
print(f'union(subject_1, subject_2) : {len(union_raw)}  (correct count)')
print()

missing_s1 = union_raw - s1_raw
missing_s2 = union_raw - s2_raw
print(f'Missing from subject_1: {missing_s1} (lex-LAST participant — always in subject_2)')
print(f'Missing from subject_2: {missing_s2} (lex-FIRST participant — always in subject_1)')
print()

all_codes = sorted(union_raw)
print('Lexicographic order of all 40 participants:')
for i, code in enumerate(all_codes):
    marker = ''
    if code in missing_s1:
        marker = '  ← always subject_2 (lex-LAST)'
    elif code in missing_s2:
        marker = '  ← always subject_1 (lex-FIRST)'
    print(f'  [{i+1:2d}] {code}{marker}')

=== pair_cache_raw (HH pairs only) ===
Total HH pairs: 264,420

subject_1.nunique()         : 39  (naive count — misleading!)
subject_2.nunique()         : 39  (naive count — misleading!)
union(subject_1, subject_2) : 40  (correct count)

Missing from subject_1: {'ZEF2XJBX'} (lex-LAST participant — always in subject_2)
Missing from subject_2: {'0S9EIHUY'} (lex-FIRST participant — always in subject_1)

Lexicographic order of all 40 participants:
  [ 1] 0S9EIHUY  ← always subject_1 (lex-FIRST)
  [ 2] 0X4B4L3T
  [ 3] 1UB7WNAM
  [ 4] 3BN3ZBUX
  [ 5] 85M7GIDP
  [ 6] 8HLCRUY1
  [ 7] 9PLAS8J9
  [ 8] BEK0Y4W3
  [ 9] BTTWUQG4
  [10] DOF4GFV3
  [11] DPCN9S37
  [12] DYAN3RF8
  [13] EN84KWE9
  [14] FG80C8QR
  [15] FLWP3ZF7
  [16] FXDCPFHM
  [17] FYXQSR7I
  [18] HB857QDA
  [19] HGIX7HE0
  [20] IM2K4RKB
  [21] IR1A0UYG
  [22] IV0PL083
  [23] J09HHJXK
  [24] J0OU3FTE
  [25] JOJCF28V
  [26] K8VD3TYB
  [27] KS8J7LVP
  [28] MPOMQQNU
  [29] NFZJ7A7N
  [30] NVF556XP
  [31] O23790AV
  [32] OBRNN1JM
  [33] 

In [9]:
# Per-question-variant coverage in pair_cache_raw
per_qv_raw = hh_raw.groupby(['question_id','variant']).apply(
    lambda x: len(set(x.subject_1) | set(x.subject_2)), include_groups=False)

print('Per-question-variant unique participant counts (raw):')
print(per_qv_raw.value_counts().sort_index())
print(f'All have 40? {(per_qv_raw == 40).all()}')

Per-question-variant unique participant counts (raw):
40    339
Name: count, dtype: int64
All have 40? True


## Stage 5 — `pair_cache_cleaned.parquet`

`build_cleaned_pair_cache()` cleans both answer strings (strips refusals, preambles, applies VQA normalisation) and **drops rows where either answer becomes empty after cleaning**. This causes one (question, variant) to drop from 40 → 39 unique participants.

In [10]:
pair_clean = pd.read_parquet(EXPORTS / 'pair_cache_cleaned.parquet')
hh_clean = pair_clean[pair_clean['pair_type'] == 'HH']

print('=== pair_cache_cleaned (HH pairs only) ===')
print(f'Total HH pairs: {len(hh_clean):,}')
print()

s1_c = set(hh_clean['subject_1'].unique())
s2_c = set(hh_clean['subject_2'].unique())
union_c = s1_c | s2_c

print(f'subject_1.nunique()         : {len(s1_c)}  (naive count — misleading!)')
print(f'subject_2.nunique()         : {len(s2_c)}  (naive count — misleading!)')
print(f'union(subject_1, subject_2) : {len(union_c)}  (correct count)')
print()

# Per-q-v coverage
per_qv_clean = hh_clean.groupby(['question_id','variant']).apply(
    lambda x: len(set(x.subject_1) | set(x.subject_2)), include_groups=False)

print('Per-question-variant unique participant counts (cleaned):')
print(per_qv_clean.value_counts().sort_index())
print()

qv_not_40 = per_qv_clean[per_qv_clean < 40]
print(f'Question-variants with < 40 participants: {len(qv_not_40)}')
if len(qv_not_40):
    for (qid, var), n in qv_not_40.items():
        print(f'  QID={qid}, variant={var}: N={n}')

=== pair_cache_cleaned (HH pairs only) ===
Total HH pairs: 264,381

subject_1.nunique()         : 39  (naive count — misleading!)
subject_2.nunique()         : 39  (naive count — misleading!)
union(subject_1, subject_2) : 40  (correct count)

Per-question-variant unique participant counts (cleaned):
39      1
40    338
Name: count, dtype: int64

Question-variants with < 40 participants: 1
  QID=353889001, variant=A: N=39


In [11]:
# Identify which participant is missing from the affected q-v and why
from analysis.build_pair_cache import clean_answer

for (qid, var), n in qv_not_40.items():
    sub = hh_clean[(hh_clean['question_id'] == qid) & (hh_clean['variant'] == var)]
    humans_present = set(sub.subject_1) | set(sub.subject_2)
    missing = union_c - humans_present
    print(f'QID={qid}, variant={var}: missing participant(s) = {missing}')
    print()

    for pid in missing:
        # Find their raw response
        raw_row = h_csv[
            (h_csv['participant'] == pid) &
            (h_csv['question_id'] == qid) &
            (h_csv['variant'] == var)
        ]
        raw_resp = raw_row['response'].iloc[0] if len(raw_row) else '(not found)'
        cleaned = clean_answer(raw_resp)
        print(f'  Participant:     {pid}')
        print(f'  Raw response:    {repr(raw_resp)}')
        print(f'  After cleaning:  {repr(cleaned)}')
        print(f'  → Empty after cleaning: {cleaned == ""}')
        print()

    # Also check the question
    q_row = h_csv[h_csv['question_id'] == qid]
    q_text = q_row[q_row['variant'] == var]['question_en'].iloc[0] if len(q_row) else ''
    print(f'  Question text (variant {var}): {q_text}')

/home/david/miniconda3/envs/zero/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


QID=353889001, variant=A: missing participant(s) = {'FG80C8QR'}

  Participant:     FG80C8QR
  Raw response:    'a'
  After cleaning:  ''
  → Empty after cleaning: True

  Question text (variant A): What letter is it?


## Summary

| Stage | File | Participants | Questions |
|-------|------|:---:|:---:|
| Raw JSON | `evaluation/humans/by_participant/*.json` | 40 | 116–134 (per batch) |
| Qualifying participants | `load_human_data()` (min_answers=348) | 40 | — |
| Common question set | intersection across all 40 | 40 | **113** |
| `responses_human.csv` | 40 × 113 × 3 = 13,560 rows | 40 | 113 |
| After `check_completeness()` | all 40 pass (all have 113 q's) | **40** | 113 |
| `pair_cache_raw` (HH) | all 40 paired | **40** (union) | 113 × 3 = 339 q-v |
| `pair_cache_cleaned` (HH) | one q-v drops to 39 due to empty clean | **40** (union) | 338 full + 1 with N=39 |

### Why pair_cache appears to have N=39

The pair cache stores pairs with **lexicographic ordering**: `subject_1 = min(s1, s2)`, `subject_2 = max(s1, s2)`. This means:
- `ZEF2XJBX` (lexicographically last of 40) → **always in `subject_2`**, never in `subject_1`
- `0S9EIHUY` (lexicographically first of 40) → **always in `subject_1`**, never in `subject_2`

So `subject_1.nunique()` = 39 and `subject_2.nunique()` = 39, but the **union is 40** (the correct N).

Additionally, participant **FG80C8QR** answered `"a"` for question 353889001 variant A. After VQA normalisation `clean_answer("a") → ""`. All their pairs for that q-v are dropped in the cleaned cache, so that single q-v has only 39 participants.

### Participant batches

The 40 participants are from 3 batches that answered different numbers of questions:
- **Batch 1** (116 questions): 11 participants — earliest sessions
- **Batch 2** (124 questions): 5 participants — intermediate sessions  
- **Batch 3** (134 questions): 24 participants — later sessions with expanded question set

The common intersection of all three batches = **113 questions** (used in all analyses).

In [12]:
# Final verification: correct way to count unique participants from pair cache
print('=== Correct participant counts from pair_cache_cleaned ===')
print()

hh = pair_clean[pair_clean['pair_type'] == 'HH']

print('WRONG (single column):')  
print(f'  hh["subject_1"].nunique() = {hh["subject_1"].nunique()}')
print(f'  hh["subject_2"].nunique() = {hh["subject_2"].nunique()}')
print()
print('CORRECT (union of both columns):')
n_correct = len(set(hh['subject_1']) | set(hh['subject_2']))
print(f'  len(set(subject_1) | set(subject_2)) = {n_correct}')
print()
print('ALSO CORRECT (from responses_human.csv):')
print(f'  h_csv["participant"].nunique() = {h_csv["participant"].nunique()}')

=== Correct participant counts from pair_cache_cleaned ===

WRONG (single column):
  hh["subject_1"].nunique() = 39
  hh["subject_2"].nunique() = 39

CORRECT (union of both columns):
  len(set(subject_1) | set(subject_2)) = 40

ALSO CORRECT (from responses_human.csv):
  h_csv["participant"].nunique() = 40
